# 47 Expense Audit Agent

**Pattern**: Configurable policy rule engine — POLICY dict injected as LLM context  
**Framework**: LangChain (`langchain-openai`) with structured output  
**Key idea**: All T&E limits live in a single `POLICY` dict. The LLM evaluates each expense line against the injected limits; nothing is hardcoded in prompts.

In [ ]:
%pip install -q langchain langchain-openai pydantic python-dotenv

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Part 1 — Business Problem

Finance teams manually review hundreds of expense reports each month, checking every line against category limits, receipt requirements, and travel-class rules. This process is:

- **Inconsistent** — different reviewers apply the policy differently.
- **Slow** — high-value reports queue for days waiting for a human reviewer.
- **Error-prone** — city-based tiered limits are easy to misremember.

**Agent design goal**: evaluate each expense line against the configured policy, produce typed `PolicyViolation` objects with rule IDs and severity, and route each report to the correct approval tier — consistently and in seconds.

**Policy as code**: limits live in a `POLICY` dict, not prompt text. Changing a limit requires editing one value in one place. The LLM receives the applicable limit for each line at runtime via context injection.

## Part 2 — Schema

Three Pydantic models drive the audit:

| Model | Purpose |
|---|---|
| `ExpenseLine` | One expense line submitted by an employee |
| `PolicyViolation` | A single rule breach detected by the LLM |
| `AuditResult` | Full audit output: violations, counts, tier, summary |

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field


class ExpenseLine(BaseModel):
    line_id: str = Field(description="Unique identifier for this expense line")
    date: str = Field(description="Expense date in YYYY-MM-DD format")
    category: Literal[
        "accommodation", "meals", "transport",
        "entertainment", "equipment", "other",
    ] = Field(description="Expense category")
    amount: float = Field(description="Amount claimed")
    city: Optional[str] = Field(default=None, description="City of the expense")
    description: str = Field(description="Employee-supplied description")
    receipt_attached: bool = Field(description="Receipt or invoice attached?")
    pre_approved: bool = Field(default=False, description="Was this pre-approved?")
    class_of_travel: Optional[Literal["economy", "premium_economy", "business", "first"]] = Field(
        default=None, description="Cabin class for transport lines"
    )


class PolicyViolation(BaseModel):
    line_id: str = Field(description="ID of the expense line that triggered this violation")
    rule_id: str = Field(description="Rule code, e.g. MEAL-001, HOTEL-002, TRAVEL-003")
    rule_description: str = Field(description="Human-readable rule description")
    violation_detail: str = Field(description="How the line breaches the rule")
    severity: Literal["info", "warn", "block"] = Field(
        description="info=note, warn=review needed, block=cannot approve"
    )


class AuditResult(BaseModel):
    report_id: str = Field(description="Expense report identifier")
    employee_name: str = Field(description="Employee who submitted the report")
    total_claimed: float = Field(description="Total amount claimed")
    violations: List[PolicyViolation] = Field(description="All violations found")
    compliant_lines: int = Field(description="Lines with no violations")
    violation_lines: int = Field(description="Lines with at least one violation")
    approval_tier: Literal[
        "auto_approve", "line_manager", "finance_director", "rejected"
    ] = Field(description="Routing decision")
    audit_summary: str = Field(description="Narrative summary for the approver")


print("Schema defined — ExpenseLine, PolicyViolation, AuditResult ready.")

## Part 3 — Policy Configuration

All T&E limits live in the `POLICY` dict — the **single source of truth**.

### Why this matters

If limits were hardcoded in the prompt (e.g. `"meals must not exceed $120 in NYC"`), changing any limit would require editing prompt text, re-testing the LLM, and risking inconsistency. With `POLICY` as code:

- Change one value → all audit runs immediately use the new limit.
- The prompt never mentions specific dollar amounts — it receives them at runtime.
- A non-engineer (e.g. the CFO) can review and approve the `POLICY` dict without reading LLM prompts.

Two helper functions translate city names into the applicable limit without exposing the tier logic to callers.

In [ ]:
POLICY = {
    "meal_limits": {
        "tier_1": {"cities": ["NYC", "SF", "London", "Tokyo", "Dubai"], "daily_limit": 120.0},
        "tier_2": {"cities": ["Chicago", "LA", "Paris", "Singapore"], "daily_limit": 90.0},
        "default": {"daily_limit": 60.0},
    },
    "accommodation_limits": {
        "tier_1": {"cities": ["NYC", "SF", "London", "Tokyo", "Dubai"], "nightly": 350.0},
        "tier_2": {"cities": ["Chicago", "LA", "Paris", "Singapore"], "nightly": 250.0},
        "default": {"nightly": 180.0},
    },
    "receipt_threshold": 25.0,
    "entertainment_limit": 200.0,
    "transport": {"requires_pre_approval": ["business", "first"]},
    "equipment_limit": 500.0,
}


def get_meal_limit(city):
    if city is None:
        return POLICY["meal_limits"]["default"]["daily_limit"]
    city_upper = city.strip().upper()
    for tier in ("tier_1", "tier_2"):
        if city_upper in [c.upper() for c in POLICY["meal_limits"][tier]["cities"]]:
            return POLICY["meal_limits"][tier]["daily_limit"]
    return POLICY["meal_limits"]["default"]["daily_limit"]


def get_accommodation_limit(city):
    if city is None:
        return POLICY["accommodation_limits"]["default"]["nightly"]
    city_upper = city.strip().upper()
    for tier in ("tier_1", "tier_2"):
        if city_upper in [c.upper() for c in POLICY["accommodation_limits"][tier]["cities"]]:
            return POLICY["accommodation_limits"][tier]["nightly"]
    return POLICY["accommodation_limits"]["default"]["nightly"]


# Smoke test
assert get_meal_limit("NYC") == 120.0
assert get_meal_limit("Chicago") == 90.0
assert get_meal_limit("Austin") == 60.0
assert get_accommodation_limit("SF") == 350.0
assert get_accommodation_limit("Paris") == 250.0
assert get_accommodation_limit("Dallas") == 180.0
print("POLICY and helpers verified.")

## Part 4 — Workflow

The `run()` function:

1. Calls `_build_policy_context()` to pair each `ExpenseLine` with its applicable limits.
2. Invokes `gpt-4.1-nano` via LangChain with `with_structured_output(AuditResult)`.
3. Applies deterministic approval routing post-LLM.

**Approval tier routing**:

| Condition | Tier |
|---|---|
| No violations | `auto_approve` |
| info/warn only | `line_manager` |
| Any block + total < $5,000 | `finance_director` |
| Any block + total >= $5,000 OR missing receipt above threshold | `rejected` |

In [ ]:
import json
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI

AUDITOR_PROMPT = SystemMessage(
    content=(
        "You are a corporate T&E expense auditor. "
        "Evaluate each expense line against the policy limits provided in the user message "
        "and generate a PolicyViolation for every breach.\n\n"
        "Severity: info=minor note, warn=approver review required, block=cannot approve as-is.\n"
        "Rule IDs: MEAL-001 HOTEL-002 TRAVEL-003 RECEIPT-001 ENT-001 EQUIP-001.\n\n"
        "Return the full AuditResult including all fields."
    )
)


def _build_policy_context(lines):
    per_line = []
    for line in lines:
        limits = {}
        if line.category == "meals":
            limits["daily_meal_limit"] = get_meal_limit(line.city)
        elif line.category == "accommodation":
            limits["nightly_accommodation_limit"] = get_accommodation_limit(line.city)
        elif line.category == "entertainment":
            limits["entertainment_limit"] = POLICY["entertainment_limit"]
        elif line.category == "equipment":
            limits["equipment_limit"] = POLICY["equipment_limit"]
        elif line.category == "transport":
            limits["requires_pre_approval_classes"] = POLICY["transport"]["requires_pre_approval"]
        limits["receipt_threshold"] = POLICY["receipt_threshold"]
        per_line.append({"line": line.model_dump(), "applicable_limits": limits})
    return json.dumps(per_line, indent=2)


def _determine_approval_tier(violations, total_claimed, lines):
    has_block = any(v.severity == "block" for v in violations)
    missing_receipt = any(
        not line.receipt_attached and line.amount > POLICY["receipt_threshold"]
        for line in lines
    )
    if not violations:
        return "auto_approve"
    if not has_block and not missing_receipt:
        return "line_manager"
    if has_block and total_claimed < 5000.0:
        return "finance_director"
    return "rejected"


def run(report_id, employee_name, lines, model="gpt-4.1-nano"):
    total_claimed = sum(line.amount for line in lines)
    policy_context = _build_policy_context(lines)
    user_message = (
        f"Expense report: {report_id}\nEmployee: {employee_name}\n"
        f"Total claimed: {total_claimed:.2f}\n\n"
        f"Expense lines with applicable policy limits:\n{policy_context}\n\n"
        f"Routing rules: no violations->auto_approve; info/warn only->line_manager; "
        f"any block + total<5000->finance_director; "
        f"block + total>=5000 OR missing receipt above {POLICY['receipt_threshold']}->rejected.\n"
        f"Return a complete AuditResult."
    )
    llm = ChatOpenAI(model=model, temperature=0, api_key=os.environ["OPENAI_API_KEY"])
    auditor = AUDITOR_PROMPT | llm.with_structured_output(AuditResult)
    result = auditor.invoke({"messages": [{"role": "user", "content": user_message}]})

    approval_tier = _determine_approval_tier(result.violations, total_claimed, lines)
    line_ids_with_violations = {v.line_id for v in result.violations}
    violation_lines = len(line_ids_with_violations)
    compliant_lines = len(lines) - violation_lines

    return AuditResult(
        report_id=report_id,
        employee_name=employee_name,
        total_claimed=total_claimed,
        violations=result.violations,
        compliant_lines=compliant_lines,
        violation_lines=violation_lines,
        approval_tier=approval_tier,
        audit_summary=result.audit_summary,
    )


print("run() defined.")

## Part 5 — Run Clean Report (Alice Chen)

All lines within policy limits, receipts attached, economy travel pre-approved.  
Expected outcome: **auto_approve**.

In [ ]:
alice_lines = [
    ExpenseLine(line_id="L01", date="2025-06-10", category="transport", amount=380.00,
                city="NYC", description="Economy flight NYC return",
                receipt_attached=True, pre_approved=True, class_of_travel="economy"),
    ExpenseLine(line_id="L02", date="2025-06-11", category="accommodation", amount=320.00,
                city="NYC", description="Hotel stay NYC 1 night", receipt_attached=True),
    ExpenseLine(line_id="L03", date="2025-06-11", category="meals", amount=95.00,
                city="NYC", description="Team working lunch", receipt_attached=True),
    ExpenseLine(line_id="L04", date="2025-06-11", category="transport", amount=35.00,
                city="NYC", description="Taxi to client office", receipt_attached=True),
]

result1 = run("EXP-2025-001", "Alice Chen", alice_lines)
print(f"Report : {result1.report_id}")
print(f"Total  : ${result1.total_claimed:,.2f}")
print(f"Tier   : {result1.approval_tier}")
print(f"Lines  : {result1.compliant_lines} compliant, {result1.violation_lines} violation(s)")
print(f"Summary: {result1.audit_summary}")

## Part 6 — Run Violations Report (Bob Kumar)

Business-class flight without pre-approval, hotel above NYC limit, meals over NYC limit, entertainment over limit, equipment over limit.  
Expected outcome: **finance_director** (blocks present, total < $5,000).

In [ ]:
bob_lines = [
    ExpenseLine(line_id="L01", date="2025-06-05", category="transport", amount=3200.00,
                city="NYC", description="Business class flight to NYC",
                receipt_attached=True, pre_approved=False, class_of_travel="business"),
    ExpenseLine(line_id="L02", date="2025-06-06", category="accommodation", amount=420.00,
                city="NYC", description="Hotel NYC above limit", receipt_attached=True),
    ExpenseLine(line_id="L03", date="2025-06-06", category="meals", amount=150.00,
                city="NYC", description="Client dinner meals over limit", receipt_attached=True),
    ExpenseLine(line_id="L04", date="2025-06-07", category="entertainment", amount=280.00,
                city="NYC", description="Client entertainment event", receipt_attached=True),
    ExpenseLine(line_id="L05", date="2025-06-08", category="equipment", amount=620.00,
                city="NYC", description="Laptop for home office", receipt_attached=True),
]

result2 = run("EXP-2025-002", "Bob Kumar", bob_lines)
print(f"Report : {result2.report_id}")
print(f"Total  : ${result2.total_claimed:,.2f}")
print(f"Tier   : {result2.approval_tier}")
print(f"Lines  : {result2.compliant_lines} compliant, {result2.violation_lines} violation(s)")
for v in result2.violations:
    print(f"  [{v.severity.upper():5}] {v.rule_id} -- {v.violation_detail}")
print(f"Summary: {result2.audit_summary}")

## Part 7 — Exercise: Adding a New City to the Policy

Your company has opened an office in **Riyadh**. The CFO wants to set:
- Daily meal limit: **$100**
- Nightly accommodation limit: **$280**

Answer these questions, then check the answer key.

**Q1**: What changes in `POLICY`?  
**Q2**: What changes in `tools.py` (the helper functions)?  
**Q3**: What changes in `prompts.py`?  
**Q4**: Why does `prompts.py` not need to change?

In [ ]:
# Your answers here
q1_policy_changes = ""    # What dict keys/values change?
q2_tools_changes = ""     # Do the helper functions change?
q3_prompts_changes = ""   # Does AUDITOR_PROMPT change?
q4_why_no_prompt_change = ""  # Explain the design principle

print("Fill in your answers above, then run the answer-key cell.")

In [ ]:
# Answer key
print("=" * 60)
print("ANSWER KEY")
print("=" * 60)
print()
print("Q1: Changes to POLICY")
print("  Add a new tier entry (e.g. 'tier_3' or 'riyadh') in both")
print("  meal_limits and accommodation_limits:")
print("  POLICY['meal_limits']['tier_3'] = {")
print("      'cities': ['Riyadh'], 'daily_limit': 100.0")
print("  }")
print("  POLICY['accommodation_limits']['tier_3'] = {")
print("      'cities': ['Riyadh'], 'nightly': 280.0")
print("  }")
print("  No other dict entries change.")
print()
print("Q2: Changes to tools.py (helper functions)")
print("  Update the loop in get_meal_limit() and")
print("  get_accommodation_limit() to iterate over the new tier key.")
print("  If using a list of known tier names, add 'tier_3' to that list.")
print("  If using POLICY.keys() dynamically, no function code changes.")
print()
print("Q3: Changes to prompts.py")
print("  None. AUDITOR_PROMPT does not mention any specific city")
print("  or dollar amount.")
print()
print("Q4: Why prompts.py does not need to change")
print("  The POLICY dict is injected as runtime context via")
print("  _build_policy_context(). The prompt is a policy-agnostic")
print("  evaluation instruction; the data (limits) is separate from")
print("  the instruction (evaluate). This is the 'policy as code'")
print("  pattern: configuration is separated from logic, so limits")
print("  can be changed, audited, and versioned independently of")
print("  the LLM prompt. Non-engineers can review the POLICY dict")
print("  without touching any prompt engineering.")